Colab is making it easier than ever to integrate powerful Generative AI capabilities into your projects. We are launching public preview for a simple and intuitive Python library (google.colab.ai) to access state-of-the-art language models directly within Colab environments. All users have free access to most popular LLMs, while paid users have access to a wider selection of models. This means users can spend less time on configuration and set up and more time bringing their ideas to life. With just a few lines of code, you can now perform a variety of tasks:
- Generate text
- Translate languages
- Write creative content
- Categorize text

Happy Coding!


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/Getting_started_with_google_colab_ai.ipynb)

In [ ]:
# @title List available models
from google.colab import ai

ai.list_models()

['google/gemini-2.0-flash',
 'google/gemini-2.0-flash-lite',
 'google/gemini-2.5-flash',
 'google/gemini-2.5-flash-lite',
 'google/gemini-2.5-pro',
 'google/gemma-3-12b',
 'google/gemma-3-1b',
 'google/gemma-3-27b',
 'google/gemma-3-4b']

Choosing a Model
The model names give you a hint about their capabilities and intended use:

Pro: These are the most capable models, ideal for complex reasoning, creative tasks, and detailed analysis.

Flash: These models are optimized for high speed and efficiency, making them great for summarization, chat applications, and tasks requiring rapid responses.

Gemma: These are lightweight, open-weight models suitable for a variety of text generation tasks and are great for experimentation.

In [5]:
# @title Simple batch generation example
# Only text-to-text input/output is supported
from google.colab import ai

response = ai.generate_text("What is the capital of France?")
print(response)

RateLimitError: Error code: 429 - {'code': '', 'message': 'The model is currently experiencing heavy load. Try again later.', 'param': '', 'type': 'rate_limit_error'}

In [ ]:
# @title Choose a different model
from google.colab import ai

response = ai.generate_text("What is the capital of England", model_name='google/gemini-2.0-flash-lite')
print(response)

For longer text generations, you can stream the response. This displays the output token by token as it's generated, rather than waiting for the entire response to complete. This provides a more interactive and responsive experience. To enable this, simply set stream=True.

In [ ]:
# @title Simple streaming example
from google.colab import ai

stream = ai.generate_text("Tell me a short story.", stream=True)
for text in stream:
  print(text, end='')

In [ ]:
#@title Text formatting setup
#code is not necessary for colab.ai, but is useful in fomatting text chunks
import sys

class LineWrapper:
    def __init__(self, max_length=80):
        self.max_length = max_length
        self.current_line_length = 0

    def print(self, text_chunk):
        i = 0
        n = len(text_chunk)
        while i < n:
            start_index = i
            while i < n and text_chunk[i] not in ' \n': # Find end of word
                i += 1
            current_word = text_chunk[start_index:i]

            delimiter = ""
            if i < n: # If not end of chunk, we found a delimiter
                delimiter = text_chunk[i]
                i += 1 # Consume delimiter

            if current_word:
                needs_leading_space = (self.current_line_length > 0)

                # Case 1: Word itself is too long for a line (must be broken)
                if len(current_word) > self.max_length:
                    if needs_leading_space: # Newline if current line has content
                        sys.stdout.write('\n')
                        self.current_line_length = 0
                    for char_val in current_word: # Break the long word
                        if self.current_line_length >= self.max_length:
                            sys.stdout.write('\n')
                            self.current_line_length = 0
                        sys.stdout.write(char_val)
                        self.current_line_length += 1
                # Case 2: Word doesn't fit on current line (print on new line)
                elif self.current_line_length + (1 if needs_leading_space else 0) + len(current_word) > self.max_length:
                    sys.stdout.write('\n')
                    sys.stdout.write(current_word)
                    self.current_line_length = len(current_word)
                # Case 3: Word fits on current line
                else:
                    if needs_leading_space:
                        # Define punctuation that should not have a leading space
                        # when they form an entire "word" (token) following another word.
                        no_leading_space_punctuation = {
                            ",", ".", ";", ":", "!", "?",        # Standard sentence punctuation
                            ")", "]", "}",                     # Closing brackets
                            "'s", "'S", "'re", "'RE", "'ve", "'VE", # Common contractions
                            "'m", "'M", "'ll", "'LL", "'d", "'D",
                            "n't", "N'T",
                            "...", "…"                          # Ellipses
                        }
                        if current_word not in no_leading_space_punctuation:
                            sys.stdout.write(' ')
                            self.current_line_length += 1
                    sys.stdout.write(current_word)
                    self.current_line_length += len(current_word)

            if delimiter == '\n':
                sys.stdout.write('\n')
                self.current_line_length = 0
            elif delimiter == ' ':
                # If line is full and a space delimiter arrives, it implies a wrap.
                if self.current_line_length >= self.max_length:
                    sys.stdout.write('\n')
                    self.current_line_length = 0

        sys.stdout.flush()


In [ ]:
# @title Formatted streaming example
from google.colab import ai

wrapper = LineWrapper()
for chunk in ai.generate_text('Give me a long winded description about the evolution of the Roman Empire.', model_name='google/gemini-2.0-flash', stream=True):
  wrapper.print(chunk)

In [6]:
import os
from google import genai
from google.genai import types
from google.colab import userdata

# 1. Получаем ключ из секретов Google Colab
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

# 2. Настраиваем агента (системная инструкция + поиск Google)
agent_config = types.GenerateContentConfig(
    system_instruction=(
            "Ты — умный, дружелюбный и эрудированный ИИ-ассистент, "
                    "работающий в интерактивном режиме в Google Colab. "
                            "Всегда используй поиск Google, когда нужны свежие факты или данные."
                                ),
                                    tools=[types.Tool(google_search=types.GoogleSearch())],
                                        temperature=0.7,
                                        )

                                        # 3. Создаем объект чата с историей
                                        # Используем модель gemini-2.5-flash (быстрая и бесплатная)
                                        chat = client.chats.create(
                                            model="gemini-2.5-flash",
                                                config=agent_config
                                                )

                                                print("=" * 60)
                                                print("🤖 ИИ-Агент успешно запущен! Напишите свое сообщение ниже.")
                                                print("Для выхода из чата введите: выход, exit или quit")
                                                print("=" * 60 + "\n")

                                                # 4. Запускаем интерактивный цикл общения
                                                while True:
                                                    # Запрос ввода от пользователя через стандартное поле Colab
                                                        user_input = input("Вы: ")

                                                                # Проверка на выход из чата
                                                                    if user_input.strip().lower() in ['выход', 'exit', 'quit', 'q']:
                                                                            print("\n🤖 Агент: До свидания! Было приятно пообщаться.")
                                                                                    break

                                                                                                if not user_input.strip():
                                                                                                        continue

                                                                                                            try:
                                                                                                                    # Отправляем сообщение в чат (история сохраняется автоматически)
                                                                                                                            response = chat.send_message(user_input)

                                                                                                                                            print(f"\n🤖 Агент: {response.text}\n")

                                                                                                                                                            # (Опционально) Показываем поисковые запросы, если агент обращался к интернету
                                                                                                                                                                    if response.candidates and response.candidates[0].grounding_metadata:
                                                                                                                                                                                metadata = response.candidates[0].grounding_metadata
                                                                                                                                                                                            if metadata.web_search_queries:
                                                                                                                                                                                                            print(f"   *[Поиск Google был использован: {metadata.web_search_queries}]*\n")

                                                                                                                                                                                                                                    print("-" * 60)

                                                                                                                                                                                                                                                except Exception as e:
                                                                                                                                                                                                                                                        print(f"\n❌ Произошла ошибка: {e}\n")
                                                                                                                                                                                                                                                                break


IndentationError: unexpected indent (4055266765.py, line 23)